# Phase 3 trainable_scope_ablation_001 — no-submit

Runs exactly two authorized variants: `freeze_backbone_train_fpn_heads` and `train_heads_only_official`. No Kaggle submission commands are present.


In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


In [ ]:
from pathlib import Path
Path('/kaggle/working/scripts').mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/artifacts/phase3_trainable_scope_ablation_001').mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile /kaggle/working/scripts/reproduce_phase3_trainable_scope_ablation.py
#!/usr/bin/env python3
"""Phase 3 no-submit trainable-scope ablation.

Protocol:
- Detectron2 DefaultTrainer on unlearn20 with empty annotations
- poisoned_model.pth initialization
- official RetinaNet config and anchors
- LR=1e-4, batch size=4, MAX_ITER=20, STEPS=[]
- trainable-scope variants only
- no L2 anchoring and no submission
- correct 16-bit preprocessing
- DefaultPredictor inference at CONF_THRESH=0.2
- writes local CSV only; never submits
"""
from __future__ import annotations

import argparse
import copy
import csv
import hashlib
import json
import math
import os
import time
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import cv2
import numpy as np
import torch

torch.backends.nnpack.enabled = False
torch.set_num_threads(int(os.environ.get("TORCH_NUM_THREADS", "1")))
torch.set_num_interop_threads(int(os.environ.get("TORCH_NUM_INTEROP_THREADS", "1")))

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, DatasetMapper, MetadataCatalog, build_detection_train_loader, detection_utils as utils
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.engine.train_loop import SimpleTrainer
from tqdm import tqdm

ROOT = Path(__file__).resolve().parents[1]
DEFAULT_CKPT = ROOT / "data/raw/poisoned_model.pth"
DEFAULT_UNLEARN_DIR = ROOT / "data/raw"
DEFAULT_TEST_DIR = ROOT / "data/test_download/extracted/test_set/test_set"
DEFAULT_SAMPLE = ROOT / "data/test_download/extracted/sample_submission.csv"
DEFAULT_OUTDIR = ROOT / "artifacts/phase3_trainable_scope_ablation_001"
EXPECTED_CKPT_SHA256 = "f6c21faa2a5b56549fc9e058147c90b149a034858fe0678f5a99ea5a6f0e657c"

BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1
UNLEARN_LR = 1e-4
UNLEARN_ITERS = 20
BATCH_SIZE = 4
CONF_THRESH = 0.2
IMG_W = IMG_H = 1024
DATASET_NAME = "neural_debris_unlearn_empty_strict"


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def read_sample(path: Path) -> List[Dict[str, str]]:
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if not rows or list(rows[0].keys()) != ["id", "image_id", "prediction_string"]:
        raise ValueError(f"bad sample schema: {list(rows[0].keys()) if rows else None}")
    return rows


def load_16bit_scaled(path: Path) -> np.ndarray:
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(path)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


def register_unlearn_empty(unlearn_dir: Path) -> List[Dict[str, object]]:
    if DATASET_NAME in DatasetCatalog.list():
        DatasetCatalog.remove(DATASET_NAME)
        MetadataCatalog.remove(DATASET_NAME)
    json_path = unlearn_dir / "annotations_coco.json"
    with open(json_path, encoding="utf-8") as f:
        coco = json.load(f)
    dicts: List[Dict[str, object]] = []
    for im in coco["images"]:
        file_name = unlearn_dir / im["file_name"]
        if not file_name.exists():
            raise FileNotFoundError(file_name)
        dicts.append({
            "file_name": str(file_name),
            "height": im["height"],
            "width": im["width"],
            "image_id": im["id"],
            "annotations": [],
        })
    DatasetCatalog.register(DATASET_NAME, lambda dicts=dicts: dicts)
    MetadataCatalog.get(DATASET_NAME).set(thing_classes=["object"])
    return dicts


class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_16bit_scaled(Path(dataset_dict["file_name"]))
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


class UnlearnTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)




def trainable_group(name: str) -> str:
    lname = name.lower()
    if lname.startswith("backbone.bottom_up"):
        return "backbone_bottom_up"
    if lname.startswith("backbone"):
        return "fpn_top_block"
    if lname.startswith("head"):
        return "retinanet_head"
    return "other"


def apply_trainable_scope(model, scope: str) -> Dict[str, object]:
    """Apply the two pre-authorized Phase 3 scopes.

    freeze_backbone_train_fpn_heads:
      freeze backbone.bottom_up.* only; train FPN/top block and full head.*
    train_heads_only_official:
      freeze all backbone.* including FPN/top block; train full head.*
    """
    if scope not in {"freeze_backbone_train_fpn_heads", "train_heads_only_official"}:
        raise ValueError(f"unauthorized trainable scope: {scope}")
    group_counts: Dict[str, Dict[str, int]] = {}
    examples = {"trainable": [], "frozen": []}
    for name, p in model.named_parameters():
        if scope == "freeze_backbone_train_fpn_heads":
            trainable = not name.startswith("backbone.bottom_up")
        elif scope == "train_heads_only_official":
            trainable = name.startswith("head")
        p.requires_grad = bool(trainable)
        group = trainable_group(name)
        g = group_counts.setdefault(group, {"total_params": 0, "trainable_params": 0, "frozen_params": 0, "tensors": 0, "trainable_tensors": 0})
        n = int(p.numel())
        g["total_params"] += n
        g["tensors"] += 1
        if trainable:
            g["trainable_params"] += n; g["trainable_tensors"] += 1
            if len(examples["trainable"]) < 20: examples["trainable"].append(name)
        else:
            g["frozen_params"] += n
            if len(examples["frozen"]) < 20: examples["frozen"].append(name)
    total = sum(g["total_params"] for g in group_counts.values())
    trainable = sum(g["trainable_params"] for g in group_counts.values())
    return {"scope": scope, "total_params": total, "trainable_params": trainable, "frozen_params": total-trainable, "groups": group_counts, "examples": examples}


def parameter_drift_vs_source(model, source_state: Dict[str, torch.Tensor]) -> Dict[str, object]:
    groups: Dict[str, Dict[str, float]] = {}
    global_diff_sq = global_src_sq = 0.0
    global_count = 0
    for name, p in model.named_parameters():
        if name not in source_state:
            continue
        src = source_state[name].to(device=p.device, dtype=p.dtype)
        diff = (p.detach() - src).float()
        srcf = src.float()
        d2 = float((diff * diff).sum().item())
        s2 = float((srcf * srcf).sum().item())
        n = int(p.numel())
        group = trainable_group(name)
        g = groups.setdefault(group, {"diff_sq":0.0,"source_sq":0.0,"count":0,"tensors":0})
        g["diff_sq"] += d2; g["source_sq"] += s2; g["count"] += n; g["tensors"] += 1
        global_diff_sq += d2; global_src_sq += s2; global_count += n
    for g in groups.values():
        g["l2"] = float(g["diff_sq"] ** 0.5)
        g["source_l2"] = float(g["source_sq"] ** 0.5)
        g["relative_l2"] = float((g["diff_sq"] ** 0.5) / max(g["source_sq"] ** 0.5, 1e-12))
        g["rmse"] = float((g["diff_sq"] / max(g["count"], 1)) ** 0.5)
    return {"global": {"diff_sq": global_diff_sq, "source_sq": global_src_sq, "count": global_count, "l2": float(global_diff_sq ** 0.5), "source_l2": float(global_src_sq ** 0.5), "relative_l2": float((global_diff_sq ** 0.5) / max(global_src_sq ** 0.5, 1e-12)), "rmse": float((global_diff_sq / max(global_count, 1)) ** 0.5)}, "groups": groups}

def make_cfg(weights: Path, outdir: Path, *, train: bool, device: str):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = str(weights)
    cfg.MODEL.DEVICE = device
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
    cfg.DATASETS.TRAIN = (DATASET_NAME,) if train else ()
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = int(os.environ.get("D2_NUM_WORKERS", "2")) if train else 0
    cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
    cfg.SOLVER.BASE_LR = UNLEARN_LR
    cfg.SOLVER.MAX_ITER = UNLEARN_ITERS
    cfg.SOLVER.STEPS = []
    cfg.OUTPUT_DIR = str(outdir)
    return cfg


def count_trainable_params(model) -> Dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_params": int(total), "trainable_params": int(trainable), "frozen_params": int(total - trainable)}


def train_variant(args) -> Path:
    t0 = time.perf_counter()
    args.outdir.mkdir(parents=True, exist_ok=True)
    ckpt_sha = sha256_file(args.checkpoint)
    if ckpt_sha != EXPECTED_CKPT_SHA256:
        raise ValueError(f"checkpoint sha mismatch: {ckpt_sha}")
    dicts = register_unlearn_empty(args.unlearn_dir)
    cfg = make_cfg(args.checkpoint, args.outdir, train=True, device=args.device)
    trainer = UnlearnTrainer(cfg)
    scope_audit = apply_trainable_scope(trainer.model, args.trainable_scope)
    param_counts = count_trainable_params(trainer.model)
    # Capture source parameters after loading poisoned checkpoint and before training.
    trainer.resume_or_load(resume=False)
    source_state = {name: p.detach().clone() for name, p in trainer.model.named_parameters()}
    # Re-apply freeze after load as a guard against checkpoint side effects.
    scope_audit = apply_trainable_scope(trainer.model, args.trainable_scope)
    param_counts = count_trainable_params(trainer.model)
    trainer.train()
    drift = parameter_drift_vs_source(trainer.model, source_state)
    final_ckpt = args.outdir / "model_final.pth"
    if not final_ckpt.exists():
        raise FileNotFoundError(final_ckpt)
    audit = {
        "mode": "phase3_trainable_scope_ablation_001_train_no_submit",
        "variant": args.trainable_scope,
        "submitted": False,
        "auto_submit": False,
        "submission_authorized": False,
        "paid_gpu_authorized": False,
        "source_checkpoint": str(args.checkpoint),
        "source_checkpoint_sha256": ckpt_sha,
        "output_checkpoint": str(final_ckpt),
        "output_checkpoint_sha256": sha256_file(final_ckpt),
        "dataset": DATASET_NAME,
        "unlearn_dir": str(args.unlearn_dir),
        "unlearn_images": len(dicts),
        "annotations_empty": True,
        "config": {
            "BASE_CONFIG": BASE_CONFIG,
            "ANCHOR_ASPECT_RATIOS": ANCHOR_ASPECT_RATIOS,
            "ANCHOR_SIZES": ANCHOR_SIZES,
            "NUM_CLASSES": NUM_CLASSES,
            "UNLEARN_LR": UNLEARN_LR,
            "UNLEARN_ITERS": UNLEARN_ITERS,
            "MAX_ITER": int(cfg.SOLVER.MAX_ITER),
            "BATCH_SIZE": BATCH_SIZE,
            "CONF_THRESH": CONF_THRESH,
            "SOLVER_STEPS": list(cfg.SOLVER.STEPS),
            "SOLVER_OPTIMIZER": "Detectron2 default SGD",
            "MODEL_DEVICE": cfg.MODEL.DEVICE,
            "DATALOADER_NUM_WORKERS": cfg.DATALOADER.NUM_WORKERS,
        },
        "scope_audit": scope_audit,
        "parameter_drift_vs_loaded_poisoned_checkpoint": drift,
        "trainable_scope": args.trainable_scope,
        "param_counts_after_build": param_counts,
        "runtime_sec": time.perf_counter() - t0,
    }
    (args.outdir / "strict_train_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    print(json.dumps(audit, indent=2), flush=True)
    return final_ckpt

def pred_string_from_instances(instances) -> str:
    out = instances.to("cpu")
    boxes = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()
    parts: List[str] = []
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, IMG_W))
        y1 = float(np.clip(y1, 0, IMG_H))
        x2 = float(np.clip(x2, 0, IMG_W))
        y2 = float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w == 0 or h == 0:
            continue
        parts.extend([f"{float(s):.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"])
    return " ".join(parts) or " "


def valid_pred_string(s: str) -> Dict[str, object]:
    vals = str(s).strip().split()
    if not vals:
        return {"valid": True, "num_detections": 0, "finite": True}
    ok_group = len(vals) % 5 == 0
    finite = True
    for v in vals:
        try:
            if not math.isfinite(float(v)):
                finite = False
        except Exception:
            finite = False
    return {"valid": ok_group and finite, "num_detections": len(vals) // 5 if ok_group else None, "finite": finite}


def infer_csv(args, checkpoint: Path) -> Path:
    t0 = time.perf_counter()
    args.outdir.mkdir(parents=True, exist_ok=True)
    rows_all = read_sample(args.sample_submission)
    rows = rows_all[args.start_index:]
    if args.max_images:
        rows = rows[: args.max_images]
    if not rows:
        raise ValueError("no rows selected")
    missing = [r["image_id"] for r in rows if not (args.test_dir / f"{r['image_id']}.png").exists()]
    if missing:
        raise FileNotFoundError(f"missing test images count={len(missing)} first={missing[:5]}")
    cfg = make_cfg(checkpoint, args.outdir, train=False, device=args.device)
    cfg.MODEL.WEIGHTS = str(checkpoint)
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
    predictor = DefaultPredictor(cfg)
    suffix = "" if args.start_index == 0 and not args.max_images else f"_part_start{args.start_index}_n{len(rows)}"
    out_csv = args.outdir / f"strict_simple_finetuning_baseline_submission{suffix}.csv"
    audit_path = args.outdir / f"strict_simple_finetuning_baseline_submission{suffix}_audit.json"

    total_det = empty = invalid = nonfinite = 0
    scores_all: List[float] = []
    widths: List[float] = []
    heights: List[float] = []
    top_scores: List[float] = []
    first20 = []
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "image_id", "prediction_string"])
        writer.writeheader()
        for idx, r in enumerate(tqdm(rows, desc=f"Strict baseline inference {args.start_index}+{len(rows)}"), start=1):
            image_id = r["image_id"]
            im = load_16bit_scaled(args.test_dir / f"{image_id}.png")
            out = predictor(im)["instances"].to("cpu")
            pred_s = pred_string_from_instances(out)
            chk = valid_pred_string(pred_s)
            vals = pred_s.strip().split()
            scores = []
            if vals:
                nums = [float(x) for x in vals]
                scores = nums[0::5]
                scores_all.extend(scores)
                widths.extend(nums[3::5])
                heights.extend(nums[4::5])
                top_scores.append(max(scores))
            nd = int(chk["num_detections"] or 0)
            total_det += nd
            empty += int(nd == 0)
            invalid += int(not chk["valid"])
            nonfinite += int(not chk["finite"])
            writer.writerow({"id": r["id"], "image_id": r["image_id"], "prediction_string": pred_s})
            if len(first20) < 20:
                first20.append({"id": r["id"], "image_id": r["image_id"], "num_detections": nd, "top_score": max(scores) if scores else None})
            if idx % 50 == 0:
                f.flush()

    def pct(vals: List[float], q: float):
        if not vals:
            return None
        return float(np.quantile(np.asarray(vals, dtype=np.float64), q))

    audit = {
        "mode": "phase3_trainable_scope_ablation_001_inference_no_submit",
        "variant": args.trainable_scope,
        "submitted": False,
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": sha256_file(checkpoint),
        "sample_submission": str(args.sample_submission),
        "test_dir": str(args.test_dir),
        "output_csv": str(out_csv),
        "output_csv_sha256": sha256_file(out_csv),
        "config": {
            "BASE_CONFIG": BASE_CONFIG,
            "ANCHOR_ASPECT_RATIOS": ANCHOR_ASPECT_RATIOS,
            "ANCHOR_SIZES": ANCHOR_SIZES,
            "NUM_CLASSES": NUM_CLASSES,
            "CONF_THRESH": CONF_THRESH,
            "device": args.device,
            "TEST_DETECTIONS_PER_IMAGE": int(cfg.TEST.DETECTIONS_PER_IMAGE),
            "RETINANET_TOPK_CANDIDATES_TEST": int(cfg.MODEL.RETINANET.TOPK_CANDIDATES_TEST),
            "RETINANET_NMS_THRESH_TEST": float(cfg.MODEL.RETINANET.NMS_THRESH_TEST),
        },
        "row_count_total_sample": len(rows_all),
        "row_start_index": args.start_index,
        "row_count": len(rows),
        "detections_total": total_det,
        "detections_per_image_mean": total_det / len(rows),
        "empty_prediction_images": empty,
        "invalid_prediction_strings": invalid,
        "nonfinite_values": nonfinite,
        "score_distribution": {"count": len(scores_all), "min": pct(scores_all, 0), "p25": pct(scores_all, 0.25), "median": pct(scores_all, 0.5), "p75": pct(scores_all, 0.75), "p95": pct(scores_all, 0.95), "max": pct(scores_all, 1)},
        "top_score_distribution": {"count": len(top_scores), "median": pct(top_scores, 0.5), "p25": pct(top_scores, 0.25), "p75": pct(top_scores, 0.75), "min": pct(top_scores, 0), "max": pct(top_scores, 1)},
        "box_width_distribution": {"count": len(widths), "median": pct(widths, 0.5), "p25": pct(widths, 0.25), "p75": pct(widths, 0.75), "min": pct(widths, 0), "max": pct(widths, 1)},
        "box_height_distribution": {"count": len(heights), "median": pct(heights, 0.5), "p25": pct(heights, 0.25), "p75": pct(heights, 0.75), "min": pct(heights, 0), "max": pct(heights, 1)},
        "per_image_summary_first20": first20,
        "runtime_sec": time.perf_counter() - t0,
    }
    audit_path.write_text(json.dumps(audit, indent=2, default=str), encoding="utf-8")
    print(json.dumps({"csv": str(out_csv), "audit": str(audit_path), "rows": len(rows), "detections_total": total_det, "empty": empty, "invalid": invalid, "nonfinite": nonfinite, "runtime_sec": audit["runtime_sec"]}, indent=2), flush=True)
    return out_csv


def main(argv: Optional[Iterable[str]] = None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", type=Path, default=DEFAULT_CKPT)
    ap.add_argument("--unlearn-dir", type=Path, default=DEFAULT_UNLEARN_DIR)
    ap.add_argument("--test-dir", type=Path, default=DEFAULT_TEST_DIR)
    ap.add_argument("--sample-submission", type=Path, default=DEFAULT_SAMPLE)
    ap.add_argument("--outdir", type=Path, default=DEFAULT_OUTDIR)
    ap.add_argument("--device", choices=["cpu", "cuda"], default="cpu")
    ap.add_argument("--train-only", action="store_true")
    ap.add_argument("--infer-only", action="store_true")
    ap.add_argument("--inference-checkpoint", type=Path, default=None)
    ap.add_argument("--trainable-scope", type=str, required=True, choices=["freeze_backbone_train_fpn_heads", "train_heads_only_official"])
    ap.add_argument("--start-index", type=int, default=0)
    ap.add_argument("--max-images", type=int, default=0)
    args = ap.parse_args(list(argv) if argv is not None else None)
    if args.train_only and args.infer_only:
        raise ValueError("choose at most one of --train-only/--infer-only")
    ckpt = args.inference_checkpoint
    if not args.infer_only:
        ckpt = train_variant(args)
    if not args.train_only:
        if ckpt is None:
            ckpt = args.outdir / "model_final.pth"
        infer_csv(args, ckpt)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# Phase 3 trainable-scope ablation: strict no-submit; only requires_grad scope changes.
import os, subprocess, sys, json, time
from pathlib import Path

os.environ.setdefault("TORCH_NUM_THREADS", "2")
os.environ.setdefault("TORCH_NUM_INTEROP_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("D2_NUM_WORKERS", "2")

ROOT = Path("/kaggle/working")
SCRIPT = ROOT / "scripts/reproduce_phase3_trainable_scope_ablation.py"
BASE_OUT = ROOT / "artifacts/phase3_trainable_scope_ablation_001"
BASE_OUT.mkdir(parents=True, exist_ok=True)

CHECKPOINT = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth")
UNLEARN_DIR = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/unlearn_set")
TEST_DIR = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/test_set/test_set")
SAMPLE = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/sample_submission.csv")

variants = ["freeze_backbone_train_fpn_heads", "train_heads_only_official"]
manifest = {
    "mode": "phase3_trainable_scope_ablation_001_no_submit",
    "submitted": False,
    "auto_submit": False,
    "submission_authorized": False,
    "variants": variants,
    "constants": {"MAX_ITER":20,"LR":1e-4,"BATCH_SIZE":4,"CONF_THRESH":0.2,"labels":"empty","preprocessing":"16-bit scaled cv2.IMREAD_UNCHANGED","detectron2":"official DefaultTrainer/DefaultPredictor","anchors":"official public ratios/sizes"},
    "runs": []
}

for scope in variants:
    outdir = BASE_OUT / scope
    outdir.mkdir(parents=True, exist_ok=True)
    log_path = outdir / f"{scope}_run.log"
    cmd = [sys.executable, str(SCRIPT), "--checkpoint", str(CHECKPOINT), "--unlearn-dir", str(UNLEARN_DIR), "--test-dir", str(TEST_DIR), "--sample-submission", str(SAMPLE), "--outdir", str(outdir), "--device", "cuda", "--trainable-scope", scope]
    print("RUN", scope, " ".join(cmd), flush=True)
    t0 = time.time()
    with open(log_path, "w", encoding="utf-8") as log:
        proc = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, text=True)
    runtime = time.time() - t0
    print("DONE", scope, "returncode", proc.returncode, "runtime_sec", runtime, flush=True)
    if proc.returncode != 0:
        print(log_path.read_text(encoding="utf-8", errors="replace")[-4000:])
        raise SystemExit(proc.returncode)
    manifest["runs"].append({"scope": scope, "outdir": str(outdir), "log": str(log_path), "runtime_sec": runtime})

(BASE_OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2), flush=True)


In [ ]:
# Build SHA256 manifest for all outputs; still no-submit.
from pathlib import Path
import hashlib, json, time
BASE_OUT = Path("/kaggle/working/artifacts/phase3_trainable_scope_ablation_001")
paths = [Path("/kaggle/working/scripts/reproduce_phase3_trainable_scope_ablation.py")]
paths.extend([p for p in BASE_OUT.rglob("*") if p.is_file()])
records=[]
for p in sorted(paths):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024), b""):
            h.update(chunk)
    records.append({"path": str(p), "relpath": str(p.relative_to(Path('/kaggle/working'))), "size": p.stat().st_size, "sha256": h.hexdigest()})
manifest={"created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "mode":"phase3_trainable_scope_ablation_001_no_submit", "submitted": False, "files": records}
(BASE_OUT / "sha256_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({"manifest": str(BASE_OUT/'sha256_manifest.json'), "file_count": len(records)}, indent=2), flush=True)


Expected outputs under `/kaggle/working/artifacts/phase3_trainable_scope_ablation_001/<scope>`: checkpoint, CSV, train/inference audits, logs, and global `sha256_manifest.json`.
